# CoffeeBench × Valto evidence review

## tl;dr

- The repository proves that Valto can intercept CoffeeBench economic tools, retain rich context and deferred outcomes, and enforce branch-specific policies without breaking the benchmark harness. All 23 local tests pass.
- The only completed LLM governed-vs-baseline branch pair is not a clean causal efficacy result: the branches diverged on day 46, while the first Valto denial occurred on day 66. Final net-income lift was **-$1,181.30**.
- Current labels are not suitable for policy ROI: zero-fill listings can be marked useful, non-dealt procurement offers can be marked useful, and all sell-side rows lack the scalar utility used by the miner.
- CoffeeBench is best kept as a long-horizon integration and policy-lifecycle case study. A hazard-dense benchmark with exact action-level utility should carry Valto's primary efficacy claim.

## Context & Methods

This notebook audits the local implementation and saved evidence as of July 13, 2026. The decision is whether CoffeeBench should be Valto's primary quantitative benchmark and, if retained, how the experiment should be redesigned.

### Key Assumptions

- `audit.annual.true_net_income` is the canonical CoffeeBench score.
- A checkpoint's last `daily_stats` row is the authoritative end-of-observation state for a stage-1 run.
- The first non-approve Valto verdict is the treatment onset for the governed branch.
- Temperature zero reduces but does not eliminate external-model nondeterminism; state divergence before treatment invalidates a single-fork 'razor-clean' causal claim.

In [1]:
from pathlib import Path
import json
import re
import sqlite3

import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'coffeebench').exists() else Path.cwd().parent
assert (ROOT / 'coffeebench').exists(), 'Notebook must live one level below the CoffeeBench repository root'
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

def read_json(relative_path):
    return json.loads((ROOT / relative_path).read_text())

def load_ledger(relative_path):
    path = ROOT / relative_path
    with sqlite3.connect(path) as con:
        frame = pd.read_sql_query(
            '''
            SELECT d.*, o.executed, o.useful, o.usefulness,
                   o.usefulness_note, o.metrics_json
            FROM decisions d
            LEFT JOIN outcomes o ON o.decision_id = d.id
            ORDER BY d.ts
            ''',
            con,
        )
    frame['context'] = frame.context_json.map(lambda value: json.loads(value or '{}'))
    frame['metrics'] = frame.metrics_json.map(lambda value: json.loads(value or '{}'))
    frame['day'] = frame.context.map(lambda value: value.get('day'))
    frame['utility'] = frame.metrics.map(lambda value: value.get('utility'))
    frame['waste'] = frame.metrics.map(lambda value: value.get('waste'))
    frame['fill_rate'] = frame.metrics.map(lambda value: value.get('fill_rate'))
    frame['dealt'] = frame.metrics.map(lambda value: value.get('dealt'))
    frame['attributed_spoilage'] = frame.metrics.map(lambda value: value.get('attributed_spoilage_w'))
    return frame


## Data

The saved artifacts mix short smoke runs, stage-1 checkpoints, deterministic validation, and one completed LLM branch comparison. This inventory prevents short smoke results from being mistaken for full-horizon benchmark evidence.

In [2]:
run_sources = {
    'deterministic_90d': 'trajectories/deterministic_economy/seed_0/run.json',
    'haiku_observe_12d': 'trajectories/roaster_valto_observe_haiku/seed_0/run.json',
    'revmax_baseline_90d': 'trajectories/roaster_valto_revmax_haiku/seed_0/valto_experiment/baseline.json',
    'revmax_governed_90d': 'trajectories/roaster_valto_revmax_haiku/seed_0/valto_experiment/governed.json',
    'gpt_easy_smoke_3d': 'trajectories/all6_gpt_valto/seed_0/run.json',
    'gpt_hard_smoke_5d': 'trajectories/all6_gpt_valto_hard/seed_0/run.json',
}

run_inventory = []
for label, path in run_sources.items():
    data = read_json(path)
    actor = data['result']['agents']['roaster_A']
    annual = actor['audit']['annual']
    run_inventory.append({
        'run': label,
        'actual_days': int(data['final_day']) + 1,
        'configured_days': data['max_days'],
        'full_horizon': int(data['final_day']) + 1 == data['max_days'],
        'roaster_A_net_income': annual['true_net_income'],
        'roaster_A_revenue': annual['true_revenue'],
        'model_cost_usd': actor['usage']['cost'],
    })
run_inventory = pd.DataFrame(run_inventory)
run_inventory

,run,actual_days,configured_days,full_horizon,roaster_A_net_income,roaster_A_revenue,model_cost_usd
0,deterministic_90d,90,90,True,2106.40,14924.7,0.000000
1,haiku_observe_12d,12,12,True,261.51,1488.0,0.674651
2,revmax_baseline_90d,90,90,True,1643.88,17050.0,9.873280
3,revmax_governed_90d,90,90,True,462.58,13126.0,8.851518
4,gpt_easy_smoke_3d,3,3,True,228.60,495.0,0.021905
5,gpt_hard_smoke_5d,5,5,True,61.11,405.0,0.085902


In [3]:
ledger_sources = {
    'haiku_observe': 'trajectories/roaster_valto_observe_haiku/seed_0/valto_ledger.db',
    'haiku_revmax_observe': 'trajectories/roaster_valto_revmax_haiku/seed_0/valto_ledger.db',
    'gpt_easy_observe': 'trajectories/all6_gpt_valto/seed_0/valto_ledger.db',
    'gpt_hard_observe': 'trajectories/all6_gpt_valto_hard/seed_0/valto_ledger.db',
    'scripted_full_surface': 'trajectories/roaster_valto_full_scripted/seed_0/valto_ledger.db',
    'revmax_governed': 'trajectories/roaster_valto_revmax_haiku/seed_0/valto_experiment/valto_ledger_governed.db',
}
ledgers = {name: load_ledger(path) for name, path in ledger_sources.items()}
decision_inventory = pd.DataFrame([
    {
        'ledger': name,
        'decisions': len(frame),
        'first_day': frame.day.min(),
        'last_day': frame.day.max(),
        'buy': int((frame.action == 'buy_green').sum()),
        'roast': int((frame.action == 'roast').sum()),
        'list': int((frame.action == 'list_roasted').sum()),
        'sell': int((frame.action == 'sell_roasted').sum()),
        'non_approve': int((frame.verdict != 'approve').sum()),
    }
    for name, frame in ledgers.items()
])
decision_inventory

,ledger,decisions,first_day,last_day,buy,roast,list,sell,non_approve
0,haiku_observe,101,1,44,64,37,0,0,0
1,haiku_revmax_observe,111,1,44,65,46,0,0,0
2,gpt_easy_observe,25,1,43,5,5,6,9,0
3,gpt_hard_observe,29,0,27,2,2,9,16,0
4,scripted_full_surface,102,0,19,37,19,19,27,0
5,revmax_governed,44,45,78,22,22,0,0,8


## Results

### The completed fork diverged before Valto intervened

Both branches start from the same day-45 checkpoint, but they re-query an external LLM. The first world-state divergence therefore matters: if it precedes the first blocked action, the final difference is not attributable solely to Valto.

In [4]:
baseline = read_json(run_sources['revmax_baseline_90d'])
governed = read_json(run_sources['revmax_governed_90d'])
governed_ledger = ledgers['revmax_governed']

def focal_daily(data):
    rows = []
    for day_row in data['daily_stats']:
        focal = day_row['per_agent']['roaster_A']
        truth = focal.get('truth_today', {})
        rows.append({
            'day': day_row['day'],
            'true_equity': focal['true_equity'],
            'cash': focal['cash'],
            'deals_count': focal['deals_count'],
            'daily_revenue': truth.get('sale_revenue', 0.0),
        })
    return pd.DataFrame(rows).set_index('day')

base_daily = focal_daily(baseline)
gov_daily = focal_daily(governed)
comparison = base_daily.join(gov_daily, lsuffix='_baseline', rsuffix='_governed')
comparison['equity_gap_governed_minus_baseline'] = (
    comparison.true_equity_governed - comparison.true_equity_baseline
)

first_world_divergence = next(
    row['day']
    for base_row, gov_row in zip(baseline['daily_stats'], governed['daily_stats'])
    if base_row != gov_row
    for row in [base_row]
)
first_intervention_day = int(governed_ledger.loc[governed_ledger.verdict != 'approve', 'day'].min())
pre_intervention_day = first_intervention_day - 1

base_actor = baseline['result']['agents']['roaster_A']
gov_actor = governed['result']['agents']['roaster_A']
base_ni = base_actor['audit']['annual']['true_net_income']
gov_ni = gov_actor['audit']['annual']['true_net_income']
final_gap = gov_ni - base_ni
pre_gap = comparison.loc[pre_intervention_day, 'equity_gap_governed_minus_baseline']
post_period_gap_change = final_gap - pre_gap

fork_summary = pd.DataFrame([
    {'metric': 'first whole-world divergence day', 'value': first_world_divergence},
    {'metric': 'first Valto non-approve verdict day', 'value': first_intervention_day},
    {'metric': 'equity gap before first intervention', 'value': round(pre_gap, 2)},
    {'metric': 'final NI gap (governed - baseline)', 'value': round(final_gap, 2)},
    {'metric': 'change in gap after treatment onset', 'value': round(post_period_gap_change, 2)},
    {'metric': 'Valto denials', 'value': int((governed_ledger.verdict == 'deny').sum())},
])
fork_summary

,metric,value
0,first whole-world divergence day,46.00
1,first Valto non-approve verdict day,66.00
2,equity gap before first intervention,-155.13
3,final NI gap (governed - baseline),-1181.30
4,change in gap after treatment onset,-1026.17
5,Valto denials,8.00


In [5]:
def truth_sum(data, entry_type, start_day=0):
    return round(sum(
        float(entry.get('amount', 0.0) or 0.0)
        for entry in data['truth_ledger']['roaster_A']
        if int(entry['day']) >= start_day and entry['entry_type'] == entry_type
    ), 2)

branch_economics = []
for arm, data in [('baseline', baseline), ('governed', governed)]:
    actor = data['result']['agents']['roaster_A']
    annual = actor['audit']['annual']
    branch_economics.append({
        'arm': arm,
        'full_run_net_income': annual['true_net_income'],
        'full_run_revenue': annual['true_revenue'],
        'post_checkpoint_revenue': truth_sum(data, 'sale_revenue', 45) - truth_sum(data, 'sale_reversal', 45),
        'post_checkpoint_cogs': truth_sum(data, 'inventory_out', 45) - truth_sum(data, 'cogs_reversal', 45),
        'post_checkpoint_spoilage': truth_sum(data, 'spoilage_expense', 45),
        'final_equity': actor['true_final_equity'],
    })
branch_economics = pd.DataFrame(branch_economics)
branch_economics

,arm,full_run_net_income,full_run_revenue,post_checkpoint_revenue,post_checkpoint_cogs,post_checkpoint_spoilage,final_equity
0,baseline,1643.88,17050.0,7240.0,5160.84,20.91,16820.28
1,governed,462.58,13126.0,3316.0,2390.53,51.22,15638.98


### The label/miner contract misses the margin problem

The miner loads only rows containing `metrics.utility`. In the current labeler, that scalar exists for procurement and roasting, but not for listing or selling. Several edge cases also assign favorable usefulness labels to behavior that should remain unresolved or negative.

In [6]:
observe_names = [
    'haiku_observe', 'haiku_revmax_observe', 'gpt_easy_observe',
    'gpt_hard_observe', 'scripted_full_surface',
]
quality_rows = []
for name in observe_names:
    frame = ledgers[name]
    listing = frame[frame.action == 'list_roasted']
    buying = frame[frame.action == 'buy_green']
    selling = frame[frame.action == 'sell_roasted']
    roasting = frame[frame.action == 'roast']
    quality_rows.append({
        'ledger': name,
        'decisions': len(frame),
        'zero_fill_listings_marked_useful': int(((listing.fill_rate == 0) & (listing.useful == 1)).sum()),
        'non_dealt_buys_marked_useful': int(((buying.dealt == False) & (buying.useful == 1)).sum()),  # noqa: E712
        'sell_rows_missing_miner_utility': int(selling.utility.isna().sum()),
        'roasts_negative_only_due_to_spoilage': int(((roasting.attributed_spoilage > 0) & (roasting.utility < 0)).sum()),
    })
label_quality = pd.DataFrame(quality_rows)
label_quality.loc['total'] = label_quality.select_dtypes('number').sum()
label_quality

,ledger,decisions,zero_fill_listings_marked_useful,non_dealt_buys_marked_useful,sell_rows_missing_miner_utility,roasts_negative_only_due_to_spoilage
0,haiku_observe,101.0,0.0,3.0,0.0,27.0
1,haiku_revmax_observe,111.0,0.0,5.0,0.0,36.0
2,gpt_easy_observe,25.0,0.0,0.0,9.0,0.0
3,gpt_hard_observe,29.0,2.0,0.0,16.0,2.0
4,scripted_full_surface,102.0,4.0,16.0,27.0,6.0
total,NaN,368.0,6.0,24.0,52.0,71.0


In [7]:
reports = {
    'gpt_easy': 'trajectories/all6_gpt_valto/seed_0/valto_experiment/LEARNED_POLICY_REPORT.md',
    'gpt_hard': 'trajectories/all6_gpt_valto_hard/seed_0/valto_experiment/LEARNED_POLICY_REPORT.md',
    'haiku_observe': 'trajectories/roaster_valto_observe_haiku/seed_0/valto_experiment/LEARNED_POLICY_REPORT.md',
    'haiku_revmax': 'trajectories/roaster_valto_revmax_haiku/seed_0/valto_experiment/LEARNED_POLICY_REPORT.md',
}
miner_rows = []
pattern = re.compile(r'Labeled decisions: (\d+) \(train (\d+) / eval (\d+), held-out=(True|False)\)')
for name, path in reports.items():
    text = (ROOT / path).read_text()
    match = pattern.search(text)
    miner_rows.append({
        'run': name,
        'labeled_rows': int(match.group(1)),
        'train_rows': int(match.group(2)),
        'eval_rows': int(match.group(3)),
        'held_out': match.group(4) == 'True',
        'rules_emitted': len(re.findall(r'\| yes \|', text)),
    })
miner_evidence = pd.DataFrame(miner_rows)
miner_evidence

,run,labeled_rows,train_rows,eval_rows,held_out,rules_emitted
0,gpt_easy,10,10,10,False,0
1,gpt_hard,4,4,4,False,0
2,haiku_observe,101,101,101,False,1
3,haiku_revmax,111,111,111,False,1


### The hard economy is a stress test, not a viable benchmark

At the beginning-of-day-45 checkpoint, all six agents in the hard configuration have lower true equity than at initialization. This does create hazards, but it removes the key benchmark distinction between a good decision and an unwinnable environment.

In [8]:
checkpoint_paths = {
    'gpt_easy_day45': 'trajectories/all6_gpt_valto/seed_0/valto_experiment/day45.checkpoint.json',
    'gpt_hard_day45': 'trajectories/all6_gpt_valto_hard/seed_0/valto_experiment/day45.checkpoint.json',
}
checkpoint_rows = []
for condition, path in checkpoint_paths.items():
    checkpoint = read_json(path)
    latest = checkpoint['daily_stats'][-1]['per_agent']
    for agent_id, state in latest.items():
        initial = float(checkpoint['agents'][agent_id]['initial_equity'])
        checkpoint_rows.append({
            'condition': condition,
            'agent': agent_id,
            'role': state['role'],
            'day44_true_equity': state['true_equity'],
            'initial_equity': initial,
            'equity_change': round(state['true_equity'] - initial, 2),
        })
checkpoint_equity = pd.DataFrame(checkpoint_rows)
checkpoint_equity.pivot(index='agent', columns='condition', values='equity_change')

condition,gpt_easy_day45,gpt_hard_day45
agent,,
farmer_A,-824.38,-1630.50
farmer_B,-665.77,-1233.50
retailer_A,-1051.00,-1569.34
retailer_B,-1246.50,-1712.25
roaster_A,-236.90,-935.15
roaster_B,512.46,-885.31


## Takeaways

1. **Do not publish a positive or negative Valto efficacy claim from the current CoffeeBench branches.** The one completed comparison is single-seed, unheld-out, and diverges before enforcement.
2. **Retain CoffeeBench as a secondary long-horizon case study.** It is strong evidence for protected-tool integration, deferred evaluation, policy versioning, checkpointing, and harm detection.
3. **Move the primary ROI claim to a hazard-dense, exact-attribution benchmark.** Taskmarket or a revised vending/procurement environment has much higher action volume and clearer per-action utility.
4. **If CoffeeBench continues, redesign around principal-agent misalignment.** Keep a revenue-pressured focal agent, but define Valto as the principal's profit/risk control. Compare ungoverned, prompt-only policy, and enforced policy on held-out seeds with a frozen policy and repeated LLM runs.
5. **Fix measurement before adding more difficulty.** Add exact lot/decision provenance, net-contribution labels for all four actions, intervention-regret metrics, and pre-treatment divergence checks.